# 01 - Simple Chat API: Stateless LLM Calls

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/01-simple-chat-api.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

1. Make basic chat completion API calls using the OpenAI Python library
2. Craft effective system prompts to control model behavior
3. Tune generation parameters (temperature, top_p) for different use cases
4. Manage token limits and understand their impact on cost and latency
5. Implement streaming responses for real-time output
6. Handle API errors gracefully with retry logic

---

**Architecture Pattern:** This is the simplest GenAI pattern -- a stateless request/response to an LLM API. Each call is independent with no memory of previous interactions.

## Setup

Install the required packages.

In [ ]:
!pip install openai --quiet

## Configuration

Set your OpenAI API key as an environment variable. If no key is found, the notebook will use mock responses so you can still follow along.

In [ ]:
import os
import time
import json
import sys
import random
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Mock responses will be used.")
    print("To use the real API, set your key: export OPENAI_API_KEY='sk-...'")
else:
    print("API key found. Using live OpenAI API.")

client = OpenAI(api_key=api_key if api_key else "sk-mock-key")
MODEL = "gpt-4o-mini"

---
## 1. Basic Chat Completion

The simplest possible LLM API call. We send a single user message and receive a completion. The `messages` array is the core data structure -- it contains a list of message objects, each with a `role` (system, user, or assistant) and `content`.

In [ ]:
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": "What is a large language model? Explain in 2 sentences."}
        ]
    )
    result = response.choices[0].message.content
    print("=== Response Metadata ===")
    print(f"Model: {response.model}")
    print(f"Usage: prompt_tokens={response.usage.prompt_tokens}, "
          f"completion_tokens={response.usage.completion_tokens}, "
          f"total_tokens={response.usage.total_tokens}")
    print(f"Finish reason: {response.choices[0].finish_reason}")
except Exception as e:
    print(f"API call failed: {e}")
    result = ("Mock: A large language model (LLM) is a neural network trained on massive "
              "amounts of text data that can understand and generate human-like text. "
              "These models use transformer architectures with billions of parameters to "
              "perform tasks like translation, summarization, and question answering.")
    print("Using mock response for demonstration.")

print(f"\n=== Generated Response ===")
print(result)

### Understanding the Request/Response Structure

Every OpenAI chat completion follows this contract: you send a `model` and a `messages` array, and you receive a response object with `choices`, `usage`, and metadata.

In [ ]:
# The request structure
request_payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello, world!"}
    ]
}

print("=== Request Payload ===")
print(json.dumps(request_payload, indent=2))

# Typical response structure
mock_response_structure = {
    "id": "chatcmpl-abc123",
    "object": "chat.completion",
    "created": 1709000000,
    "model": "gpt-4o-mini-2024-07-18",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "Hello! How can I help you today?"
            },
            "finish_reason": "stop"
        }
    ],
    "usage": {
        "prompt_tokens": 18,
        "completion_tokens": 9,
        "total_tokens": 27
    }
}

print("\n=== Response Structure ===")
print(json.dumps(mock_response_structure, indent=2))

---
## 2. System Prompts

System prompts set the behavior, personality, and constraints of the model. They are the primary way to control what the model does and how it responds. The system message is always the first element in the `messages` array.

In [ ]:
personas = {
    "technical_writer": (
        "You are a senior technical writer. Explain concepts clearly and concisely. "
        "Use bullet points and examples. Avoid jargon unless you define it first."
    ),
    "socratic_tutor": (
        "You are a Socratic tutor. Never give direct answers. Instead, ask guiding "
        "questions that help the student discover the answer themselves. Be encouraging."
    ),
    "code_reviewer": (
        "You are a senior software engineer doing a code review. Be constructive but "
        "thorough. Point out bugs, style issues, and suggest improvements. Use a professional tone."
    )
}

user_question = "What is recursion in programming?"

for name, system_prompt in personas.items():
    print(f"\n{'='*60}")
    print(f"Persona: {name}")
    print(f"{'='*60}")

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_question}
            ],
            max_tokens=150
        )
        result = response.choices[0].message.content
    except Exception as e:
        print(f"API call failed: {e}")
        mock_responses = {
            "technical_writer": (
                "Mock: **Recursion** is a programming technique where a function "
                "calls itself to solve a problem.\n\n"
                "- **Base case**: The condition that stops the recursion\n"
                "- **Recursive case**: The function calls itself with a modified input\n\n"
                "Example: Computing factorial -- `factorial(5) = 5 * factorial(4)`"
            ),
            "socratic_tutor": (
                "Mock: Great question! Let me ask you this -- have you ever looked "
                "at two mirrors facing each other? What happens to the reflections? "
                "How might that relate to a function that calls itself?"
            ),
            "code_reviewer": (
                "Mock: When I see recursion in code reviews, I check for: "
                "1) Is there a clear base case? Missing base cases cause stack overflows. "
                "2) Does the input converge toward the base case? "
                "3) Would an iterative approach be more readable or performant here?"
            )
        }
        result = mock_responses[name]
        print("Using mock response for demonstration.")

    print(result)

---
## 3. Temperature and Top-p Tuning

These parameters control the randomness and creativity of the model output:

- **Temperature** (0.0--2.0): Lower values produce more deterministic output. Higher values produce more creative, varied output.
- **Top-p** (0.0--1.0): Nucleus sampling -- only considers tokens within the top-p cumulative probability mass.

**Rule of thumb:** Adjust temperature OR top_p, but generally not both at once.

In [ ]:
prompt = "Write a one-sentence tagline for a coffee shop."

temperature_settings = [0.0, 0.7, 1.5]

for temp in temperature_settings:
    print(f"\n--- Temperature: {temp} ---")
    results = []
    for i in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
                max_tokens=50
            )
            results.append(response.choices[0].message.content)
        except Exception as e:
            mock_data = {
                0.0: [
                    "Where every cup tells a story.",
                    "Where every cup tells a story.",
                    "Where every cup tells a story."
                ],
                0.7: [
                    "Wake up to your best self, one sip at a time.",
                    "Where every cup tells a story.",
                    "Brewing moments that matter, every single morning."
                ],
                1.5: [
                    "Chaos in a cup, but the beautiful kind.",
                    "Your grandma's attic meets a beaker of liquid sunshine.",
                    "Fuel for dreamers, schemers, and everything between."
                ]
            }
            results.append(mock_data[temp][i])

    for j, r in enumerate(results):
        print(f"  Attempt {j+1}: {r}")

print("\nNotice: temp=0 produces identical outputs. Higher temps produce more variety.")

In [ ]:
# Top-p comparison
print("=== Top-p Comparison ===")
print("Top-p restricts the token pool the model samples from.\n")

for top_p_val in [0.1, 0.5, 1.0]:
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "Describe the color blue in one creative sentence."}],
            temperature=1.0,
            top_p=top_p_val,
            max_tokens=50
        )
        result = response.choices[0].message.content
    except Exception as e:
        print(f"API call failed: {e}")
        mock_data = {
            0.1: "Mock: Blue is the color of the sky on a clear day.",
            0.5: "Mock: Blue is the quiet hum of the ocean whispering secrets to the shore.",
            1.0: "Mock: Blue is a melancholy saxophone solo drifting through rain-slicked city streets at 3am."
        }
        result = mock_data[top_p_val]
        print("Using mock response for demonstration.")

    print(f"top_p={top_p_val}: {result}")

---
## 4. Token Limits

Understanding token limits is critical for cost control and avoiding truncated outputs. Each model has a **context window** (maximum input + output tokens). The `max_tokens` parameter caps the output length.

In [ ]:
# Token limit reference
model_limits = {
    "gpt-4o-mini": {"context_window": 128000, "max_output": 16384, "cost_input_1M": 0.15, "cost_output_1M": 0.60},
    "gpt-4o":      {"context_window": 128000, "max_output": 16384, "cost_input_1M": 2.50, "cost_output_1M": 10.00},
    "gpt-4-turbo": {"context_window": 128000, "max_output": 4096,  "cost_input_1M": 10.00, "cost_output_1M": 30.00},
}

print(f"{'Model':<15} {'Context':>10} {'Max Output':>12} {'Input $/1M':>12} {'Output $/1M':>12}")
print("-" * 65)
for model_name, info in model_limits.items():
    print(f"{model_name:<15} {info['context_window']:>10,} {info['max_output']:>12,} "
          f"${info['cost_input_1M']:>10.2f} ${info['cost_output_1M']:>10.2f}")

In [ ]:
# Demonstrating max_tokens effect on output
prompt = "Write a detailed explanation of how neural networks learn."

for max_tok in [30, 100, 300]:
    print(f"\n--- max_tokens={max_tok} ---")
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tok
        )
        result = response.choices[0].message.content
        finish = response.choices[0].finish_reason
        tokens_used = response.usage.completion_tokens
        print(f"Finish reason: {finish} | Tokens used: {tokens_used}")
    except Exception as e:
        print(f"API call failed: {e}")
        mock_data = {
            30: ("Mock: Neural networks learn through a process called backpropagation, "
                 "where the model adjusts its internal weights based on", "length", 30),
            100: ("Mock: Neural networks learn through a process called backpropagation. "
                  "During training, data is passed forward through layers of neurons, each "
                  "applying weights and activation functions. The output is compared to the "
                  "expected result using a loss function. The error is then propagated backwards "
                  "through the network, and gradient descent adjusts the weights to minimize the loss.",
                  "stop", 65),
            300: ("Mock: Neural networks learn through an iterative process involving forward "
                  "propagation, loss calculation, and backpropagation. During forward propagation, "
                  "input data passes through layers of interconnected neurons, each applying learned "
                  "weights and bias terms, followed by a non-linear activation function like ReLU or "
                  "sigmoid. The final layer produces a prediction which is compared against the true "
                  "label using a loss function such as cross-entropy or mean squared error. The gradient "
                  "of this loss with respect to each weight is computed using the chain rule -- this is "
                  "backpropagation. An optimizer like SGD or Adam then updates each weight in the direction "
                  "that reduces the loss. Over many iterations (epochs) across the training data, the "
                  "network converges to weights that generalize well to unseen data.",
                  "stop", 148)
        }
        result, finish, tokens_used = mock_data[max_tok]
        print(f"Finish reason: {finish} | Tokens used: {tokens_used}")
        print("Using mock response for demonstration.")

    print(result[:250] + ("..." if len(result) > 250 else ""))

---
## 5. Streaming Responses

Streaming allows you to display tokens as they are generated, providing a much better user experience for longer outputs. Set `stream=True` and iterate over the returned chunks. Each chunk contains a small `delta` with the next piece of content.

In [ ]:
print("=== Streaming Response ===")
print("Tokens arrive incrementally:\n")

collected_chunks = []

try:
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": "List 5 benefits of exercise in brief bullet points."}
        ],
        max_tokens=200,
        stream=True
    )

    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            token = chunk.choices[0].delta.content
            collected_chunks.append(token)
            sys.stdout.write(token)
            sys.stdout.flush()

    print("\n\n--- Stream complete ---")
    full_response = "".join(collected_chunks)
    print(f"Total chunks received: {len(collected_chunks)}")

except Exception as e:
    print(f"Streaming failed: {e}")
    mock_text = (
        "- **Improved cardiovascular health**: Regular exercise strengthens the heart "
        "and improves circulation.\n"
        "- **Better mental health**: Physical activity releases endorphins, reducing "
        "stress and anxiety.\n"
        "- **Weight management**: Exercise burns calories and boosts metabolism.\n"
        "- **Stronger muscles and bones**: Resistance training increases bone density "
        "and muscle mass.\n"
        "- **Better sleep quality**: Regular physical activity helps you fall asleep "
        "faster and sleep deeper."
    )
    print("(Simulating streaming with mock data)\n")
    words = mock_text.split(" ")
    for word in words:
        sys.stdout.write(word + " ")
        sys.stdout.flush()
        time.sleep(0.04)
    print("\n\n--- Mock stream complete ---")
    print("Using mock response for demonstration.")

---
## 6. Error Handling and Retries

Production applications must handle API errors gracefully. Common failure modes:

| Error | HTTP Code | Strategy |
|---|---|---|
| Rate limit | 429 | Exponential backoff with jitter |
| Server error | 500/503 | Retry with backoff |
| Timeout | -- | Set timeout, retry |
| Auth error | 401 | Fail fast, do not retry |
| Bad request | 400 | Fail fast, fix the request |

In [ ]:
def call_with_retry(client, messages, model=MODEL, max_retries=3, base_delay=1.0):
    """Call the OpenAI API with exponential backoff retry logic."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=200
            )
            return {
                "success": True,
                "content": response.choices[0].message.content,
                "attempts": attempt + 1
            }
        except Exception as e:
            error_type = type(e).__name__
            print(f"  Attempt {attempt + 1}/{max_retries} failed: [{error_type}] {e}")

            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
                print(f"  Retrying in {delay:.1f}s...")
                time.sleep(min(delay, 2.0))  # Capped for demo

    return {
        "success": False,
        "content": None,
        "attempts": max_retries
    }


print("=== Testing Retry Logic ===")
result = call_with_retry(
    client,
    messages=[{"role": "user", "content": "Say 'retry test passed' in exactly those words."}]
)

if result["success"]:
    print(f"\nSuccess after {result['attempts']} attempt(s): {result['content']}")
else:
    print(f"\nAll {result['attempts']} attempts failed. Using fallback.")
    print("Mock: retry test passed")
    print("Using mock response for demonstration.")

In [ ]:
def safe_completion(client, messages, model=MODEL, **kwargs):
    """Production-ready wrapper for OpenAI chat completions with full metadata."""
    start_time = time.time()

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            **kwargs
        )
        elapsed = time.time() - start_time
        return {
            "status": "success",
            "content": response.choices[0].message.content,
            "model": response.model,
            "tokens": {
                "prompt": response.usage.prompt_tokens,
                "completion": response.usage.completion_tokens,
                "total": response.usage.total_tokens
            },
            "latency_ms": round(elapsed * 1000),
            "finish_reason": response.choices[0].finish_reason
        }
    except Exception as e:
        elapsed = time.time() - start_time
        return {
            "status": "error",
            "error_type": type(e).__name__,
            "error_message": str(e),
            "content": None,
            "latency_ms": round(elapsed * 1000)
        }


result = safe_completion(
    client,
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=50
)

print("=== Safe Completion Result ===")
if result["status"] == "success":
    print(json.dumps(result, indent=2))
else:
    print(f"Error: {result['error_type']}: {result['error_message']}")
    mock_result = {
        "status": "success",
        "content": "2 + 2 = 4",
        "model": "gpt-4o-mini-2024-07-18",
        "tokens": {"prompt": 14, "completion": 7, "total": 21},
        "latency_ms": 342,
        "finish_reason": "stop"
    }
    print("\nUsing mock response for demonstration.")
    print(json.dumps(mock_result, indent=2))

---
## Summary

In this notebook we covered the foundational pattern: **stateless LLM API calls**.

| Concept | Key Takeaway |
|---|---|
| Basic completion | Send messages array, get a response with metadata |
| System prompts | Control model behavior and persona via the system role |
| Temperature/top_p | Trade off between determinism and creativity |
| Token limits | Manage cost and output length with `max_tokens` |
| Streaming | Use `stream=True` for real-time token delivery |
| Error handling | Always wrap API calls with retry logic and fallbacks |

**Next:** In notebook 02, we extend this pattern to support multi-turn conversations with memory management.